# Comment Classification: LLM Models

Before running this notebook:
1. Install Ollama: https://ollama.com
2. To pull the models, run in terminal:
    - ollama pull qwen3:8b
    - ollama pull llama3.1:8b
    - ollama pull MichelRosselli/apertus:8b-instruct-2509-q4_k_m
    - ollama pull qwen3.8:27b-q4_K_M   (final model, used in analyses)

In [1]:
# importing libraries

import pandas as pd
import numpy as np
import ollama
from ollama import Client
import json
import re
from tqdm import tqdm
import krippendorff
from sklearn.metrics import classification_report, cohen_kappa_score

In [5]:
# loading the complete comments dataset (all comments, needed for parent texts)
data_cpath = "data/comments/comments.csv"
data_comments = pd.read_csv(data_cpath, encoding="latin1")

# loading the 150 manually coded comments for evaluation
data_epath = "data/comments/comments_eval.xlsx"
data_eval = pd.read_excel(data_epath, engine="openpyxl")
print(f"{len(data_eval)} rows in the evaluation set")

150 rows in the evaluation set


In [7]:
# cap human-coded breadth at 3: the codebook defines breadth as 0-3 (3 = three or more topics),
# the model can only output 0-3, but the manual coding counted up to 8 topics
data_eval["breadth"] = data_eval["breadth"].clip(upper=3)

In [10]:
# loading the system prompt (contains the full codebook) from file
with open("data/comments/system_prompt.txt", "r", encoding="utf-8") as f:
    SYSTEM_PROMPT = f.read()

In [11]:
# sanity check: is the correct (final) system prompt loaded?
print(len(SYSTEM_PROMPT))                           # length of the prompt in characters
print("LÄNGE IST KEIN BEWEIS" in SYSTEM_PROMPT)     # marker ("length is not proof") only present in the final version
print(SYSTEM_PROMPT[:300])                          # first lines for a visual check

29830
True
Du kodierst deutschsprachige Nutzerkommentare für eine sozialwissenschaftliche Studie mit
dem Titel "Likes or Dislikes, Gratifications or Concerns?", zum deutschen 9-Punkte-Plan
gegen Terrorismus ("9-Punkte-Plan") bzw. zur Terrorismuspolitik allgemein. Die Analyseeinheit
ist ein einzelner Nutzerkomm


In [12]:
# define classifier function with qwen3.8:27b as default

client = Client(timeout=300)

def classify_comment(text, parent_text=None, model="qwen3.8:27b-q4_K_M"):
    """Classify a single comment on all 6 codebook variables in one call.
    Returns a dict of 6 ints, or a dict of -1s if parsing fails."""
    fallback = {"pers_exp": -1, "emot_exp": -1, "pol_opin": -1,
                "breadth": -1, "valence": -1, "contr": -1}

    if not isinstance(text, str) or text.strip() == "":
        return {"pers_exp": 0, "emot_exp": 0, "pol_opin": 0,
                "breadth": 0, "valence": 4, "contr": 0}

    user_msg = f"Comment: {text}"
    if parent_text:
        user_msg += f"\n\n(This is a reply to the following parent comment: {parent_text})"

    try:
        response = client.chat(
            model=model,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_msg}
            ],
            format="json",
            think=False,
            keep_alive="30m",
            options={"temperature": 0, "num_ctx": 16384, "num_predict": 400}
        )
    except Exception as e:
        print(f"Error: {e}")
        return fallback

    out = response["message"]["content"].strip()

    match = re.search(r"\{.*\}", out, re.DOTALL)
    if not match:
        return fallback

    try:
        parsed = json.loads(match.group(0))
    except json.JSONDecodeError:
        return fallback

    result = {}
    bounds = {"pers_exp": (0,1), "emot_exp": (0,1), "pol_opin": (0,1),
              "breadth": (0,3), "valence": (1,4), "contr": (0,1)}
    for key, (lo, hi) in bounds.items():
        val = parsed.get(key, -1)
        try:
            val = int(val)
        except (TypeError, ValueError):
            val = -1
        result[key] = val if lo <= val <= hi else -1

    return result

In [13]:
# post_number resets per topic, so it's not unique on its own - build a composite key instead
def make_id(topic, post_number):
    return f"{topic}_{int(post_number)}"

# remove the one exact duplicate in comments.csv (ralf_md, VM3, post 38)
data_comments = data_comments.drop_duplicates(subset=["vm", "topic", "post_number", "user", "raw"]).copy()

data_comments["row_id"] = data_comments.apply(lambda r: make_id(r["topic"], r["post_number"]), axis=1)
data_eval["row_id"] = data_eval.apply(lambda r: make_id(r["topic"], r["post_number"]), axis=1)
id_col = "row_id"

# parent lookup is built from the raw forum exports (VM1-3), because comments.csv does not
# contain the July seed posts - without them, 52 replies to seed posts would have no parent text
all_posts = pd.concat(
    [pd.read_csv(f"data/comments/data_posts_{vm}.csv", encoding="cp1252") for vm in ["VM1", "VM2", "VM3"]],
    ignore_index=True
).dropna(subset=["post_number"])
all_posts["row_id"] = all_posts.apply(lambda r: make_id(r["topic"], r["post_number"]), axis=1)
text_by_id = dict(zip(all_posts["row_id"], all_posts["raw"].fillna("")))

reply_col = "reply_to_post_number"

def get_parent_text(row):
    if pd.notna(row.get(reply_col)):
        parent_id = make_id(row["topic"], row[reply_col])
        return text_by_id.get(parent_id)
    return None

# check: does every reply find its parent?
replies = data_comments[data_comments[reply_col].notna()]
n_found = replies.apply(get_parent_text, axis=1).notna().sum()
print(f"replies: {len(replies)} | parent found: {n_found}")

data_comments_full = data_comments.copy()  # keep the unfiltered version for the full run later

# only classify comments that have a manual code (validation pass)
data_comments = data_comments[data_comments[id_col].isin(data_eval[id_col])].copy()
print(f"Classifying {len(data_comments)} comments")

replies: 305 | parent found: 305
Classifying 150 comments


In [ ]:
# models to evaluate (final model: qwen3.8:27b-q4_K_M)
MODELS = ["qwen3.8:27b-q4_K_M"]

VARS = ["pers_exp", "emot_exp", "pol_opin", "breadth", "valence", "contr"]

# set to True for a quick test run with 5 comments
TEST_RUN = False
if TEST_RUN:
    data_comments = data_comments.head().copy()

texts = data_comments["raw"].fillna("")

for model_name in MODELS:
    results = []
    for i, row in tqdm(data_comments.iterrows(), total=len(data_comments), desc=f"Classifying ({model_name})"):
        parent_text = get_parent_text(row)
        codes = classify_comment(texts.loc[i], parent_text=parent_text, model=model_name)
        results.append(codes)

    results_df = pd.DataFrame(results)
    model_tag = model_name.replace(":", "_").replace(".", "_")
    pred_cols = [f"{v}_{model_tag}" for v in VARS]
    data_comments[pred_cols] = results_df.values

    # save after each model, so a kernel crash does not lose finished runs
    data_comments.to_excel(f"data/comments/eval_predictions_{model_tag}.xlsx", index=False)
    print(f"saved: eval_predictions_{model_tag}.xlsx")

Classifying (qwen3.8:27b-q4_K_M):   0%|          | 0/135 [00:30<?, ?it/s]


KeyboardInterrupt: 

In [14]:
# quick test call with a codebook example
# expected (codebook): pers_exp 0, emot_exp 0, pol_opin 1, breadth 1, valence 2, contr 1
r = client.chat(
    model="qwen3.8:27b-q4_K_M",
    messages=[{"role": "system", "content": SYSTEM_PROMPT},
              {"role": "user", "content": "Comment: Heisse Luft, wie Alles, was von Merkel kommt."}],
    format="json",
    think=False,
    options={"temperature": 0, "num_ctx": 16384, "num_predict": 400}
)
print(repr(r["message"]["content"]))

'{"pers_exp": 0, "emot_exp": 0, "pol_opin": 1, "breadth": 1, "valence": 2, "contr": 1}'


In [16]:
# parse failures (-1) per model: failed comments and failed cells per variable
for model_name in MODELS:
    tag = model_name.replace(":", "_").replace(".", "_")
    cols = [f"{v}_{tag}" for v in VARS]
    failed = (data_comments[cols] == -1)
    print(f"{model_name}: {failed.any(axis=1).sum()} of {len(data_comments)} comments with at least one -1")
    print(failed.sum().to_string(), "\n")

# first 10 predictions of the final model
final_tag = "qwen3.8:27b-q4_K_M".replace(":", "_").replace(".", "_")
print(data_comments[[f"{v}_{final_tag}" for v in VARS]].head(10))

NameError: name 'MODELS' is not defined

In [ ]:
# create report

VARS = ["pers_exp", "emot_exp", "pol_opin", "breadth", "valence", "contr"]
var_weights = {"pers_exp": None, "emot_exp": None, "pol_opin": None, "contr": None,
               "breadth": "linear", "valence": None}
var_levels = {"pers_exp": "nominal", "emot_exp": "nominal", "pol_opin": "nominal",
              "contr": "nominal", "breadth": "ordinal", "valence": "nominal"}

def krippendorff_alpha(y_true, y_pred, level):
    try:
        data = np.array([y_true, y_pred], dtype=float)
        return krippendorff.alpha(reliability_data=data, level_of_measurement=level)
    except (ZeroDivisionError, ValueError):
        return float("nan")

data_eval_renamed = data_eval.rename(columns={v: f"{v}_true" for v in VARS})
merged = data_eval_renamed.merge(data_comments, on=id_col)
print(f"Matched {len(merged)} of {len(data_eval)} rows\n")

all_rows = []

for model_name in MODELS:
    model_tag = model_name.replace(":", "_").replace(".", "_")
    print(f"\n########## {model_name} ##########")
    for var in VARS:
        y_true = merged[f"{var}_true"]
        y_pred = merged[f"{var}_{model_tag}"]

        mask = y_true.notna() & y_pred.notna() & (y_pred != -1)
        y_true_m, y_pred_m = y_true[mask], y_pred[mask]

        print(f"=== {var} (n={mask.sum()}) ===")
        report_dict = classification_report(y_true_m, y_pred_m, zero_division=0, output_dict=True)
        print(classification_report(y_true_m, y_pred_m, zero_division=0))

        kappa = cohen_kappa_score(y_true_m, y_pred_m, weights=var_weights[var])
        alpha = krippendorff_alpha(y_true_m.values, y_pred_m.values, var_levels[var])
        print(f"Cohen's Kappa: {kappa:.3f}   Krippendorff's alpha ({var_levels[var]}): {alpha:.3f}\n")

        for class_label, metrics in report_dict.items():
            if isinstance(metrics, dict):  # skips "accuracy", which is a bare float
                all_rows.append({
                    "model": model_name, "variable": var, "n": mask.sum(),
                    "class": class_label,
                    "precision": metrics["precision"], "recall": metrics["recall"],
                    "f1_score": metrics["f1-score"], "support": metrics["support"],
                    "accuracy": round(report_dict["accuracy"], 3),
                    "cohens_kappa": round(kappa, 3),
                    "krippendorff_alpha": round(alpha, 3),   # NaN stays empty in Excel
                })

report_df = pd.DataFrame(all_rows)
report_df.to_excel("data/comments/eval_report_coding_v10.xlsx", index=False)
print("Saved eval_report_coding_v10.xlsx")

# summary: one row per variable, checked against the targets (accuracy >= .80, kappa >= .60)
summary = (report_df.groupby(["model", "variable"])
           [["n", "accuracy", "cohens_kappa", "krippendorff_alpha"]].first()
           .reindex(VARS, level="variable"))
summary["meets_target"] = (summary["accuracy"] >= .80) & (summary["cohens_kappa"] >= .60)
print("\n", summary.to_string())

In [ ]:
# save the merged table (human codes "_true" next to the model predictions) for error analysis:
# open it in Excel and filter rows where a variable's "_true" and prediction columns differ,
# to see which comments the model gets wrong and why
merged.to_excel("data/comments/errors_report_v10.xlsx", index=False)
print("Saved errors_report_v10.xlsx")

gespeichert
